# Demo - Storing and Querying Vector Embeddings Using ChromaDB

Import the required packages

In [1]:
from google import genai
from google.genai import types

Set the API Key

In [2]:
from google.colab import userdata

api_key = userdata.get('GOOGLE_API_KEY')

if not api_key:
  raise RuntimeError(
      "GOOGLE_API_KEY not found. "
      "Add it via Colab -> 🔑 Secrets -> enable Notebook access."
  )

client = genai.Client(api_key=api_key)

Generate the Embeddings

In [3]:
texts = [
    "How do I write a Python function?",
    "How can I create a class in Java?",
    "Which smartphone has the best camera?",
    "What is the best Android phone?",
    "Turn off the lights.",
    "Switch the lights off.",
    "Apple released a new iPhone",
    "Apple is my favorite fruit.",
]

# By Default Dimension = 3072 but you can control dimentions using output_dimensionality=768, 1536

result = client.models.embed_content(
    model="gemini-embedding-001",
    contents=texts,
    config=types.EmbedContentConfig(task_type="SEMANTIC_SIMILARITY", outputDimensionality=768)
)

for text, embedding_obj in zip(texts, result.embeddings):
  embedding = embedding_obj.values
  print("Text:", text)
  print("Embedding length:", len(embedding))
  print("Embeddings:", embedding)
  print("-" * 50)

Text: How do I write a Python function?
Embedding length: 768
Embeddings: [-0.007334515, -0.006950146, 0.018683976, -0.04260659, -0.013810179, -0.02353132, 0.0017563101, 0.012470875, 0.03482735, 0.007981162, -0.021974491, -0.008932356, -0.03219012, 0.008530096, 0.13476881, -0.009113698, -0.023886729, 0.01596416, 0.005160837, -0.011251521, 0.0020869148, 0.015941188, -0.004785726, -0.020768173, -0.009616656, -0.015237897, 0.0153389685, 0.022661485, 0.036309812, -0.022520773, 0.0061276644, 0.02676682, 0.009503948, 0.015846258, -0.003522161, 0.013055647, 0.01262321, 0.005883274, 0.014447721, 0.04268203, -0.02048863, -0.008275668, 0.0029366165, -0.0018869105, 0.021970829, 0.005502179, -0.006817169, -0.022217361, -0.0030909744, 0.009136699, -0.00038050817, -0.0068712486, -0.011236744, -0.17840478, 0.008931584, -0.0010842282, -0.0068541258, 0.01222939, 0.0003866231, 0.0074349977, -0.023397373, 0.03202773, -0.033427857, -0.032966875, 0.004789494, -0.010730537, 0.03292936, -0.022680353, -0.0020

## 1. Install Chroma

In [4]:
pip install chromaDB

## 2. Store Embeddings in ChromaDB

In [9]:
import chromadb
from chromadb.config import Settings

# Create Chroma client (local persistent DB)
chroma_client = chromadb.Client(
    Settings(
        persist_directory="./chroma_db"
    )
)

# Create a Chroma DB collection, collection is like a table in DB
collection = chroma_client.get_or_create_collection(
    name="chroma-db-demo"
)

# Extract embeddings
embeddings = []
ids = []

for i, embedding_obj in enumerate(result.embeddings):
  embeddings.append(embedding_obj.values)
  ids.append(f"id_{i}")

# Store in Chroma
collection.add(
    ids=ids,
    documents=texts,
    embeddings=embeddings,
)

print("Embeddings stored successfully!")

Embeddings stored successfully!


## 3. Query Embeddings in ChromaDB
texts = [
    "How do I write a Python function?",
    "How can I create a class in Java?",
    "Which smartphone has the best camera?",
    "What is the best Android phone?",
    "Turn off the lights.",
    "Switch the lights off.",
    "Apple released a new iPhone",
    "Apple is my favorite fruit.",
]

In [14]:
query = "electronics"

# Generate the query_embedding using the same embedding model and dimensions
query_embedding = client.models.embed_content(
    model="gemini-embedding-001",
    contents=[query],
    config=types.EmbedContentConfig(
        task_type="SEMANTIC_SIMILARITY",
        outputDimensionality=768
    )
).embeddings[0].values

results = collection.query(
    query_embeddings=[query_embedding],
    n_results=2
)

print(results["documents"])

[['Apple released a new iPhone', 'Which smartphone has the best camera?']]
